# Task 6 — Geographic Coverage and Sampling-Bias Characterization

Characterize where historical scholarship datasets concentrate in environmental signature space, relative to the global basin distribution. Uses the Task 5 k-means clusters (k=20, Band A+B+C) as the coordinate system.

**Datasets**:
- Global: 190,675 L8 basins (the baseline)
- D-PLACE: 6,408 societies with lat/lon (ethnographic/anthropological coverage)
- WH Cities: 258 World Heritage Cities (UNESCO-designated urban heritage sites)

**Key questions**: Which environmental types are over- or under-represented in the scholarship record? Where does coverage have statistical power for correspondence testing, and where does it not?

Cliopatria excluded: spatially large temporally-smeared polygons are not comparable to point-located sites for this analysis.

See `docs/edop/data_exploration.md` Task 6 for full specification.

In [ ]:
# Cell 1 — Imports and connection
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import os, sys

sys.path.insert(0, '/Users/karlg/Documents/Repos/_cedop')
from scripts.shared.db_utils import db_connect

conn = db_connect()
out_dir = '/Users/karlg/Documents/Repos/_cedop/output/edop/explore'
print("connected")

In [ ]:
# Cell 2 — Load Task 5 cluster assignments; define cluster labels
#
# Labels derived from cluster summary (05_kmeans_cluster_summary.csv):
# sorted cold→warm by temp_yr, key discriminating variables noted.

assignments = pd.read_csv(f'{out_dir}/05_cluster_assignments.csv')
basin_to_cluster = dict(zip(assignments.hybas_id.astype(int),
                            assignments.km_cluster.astype(int)))

CLUSTER_LABELS = {
    19: "Arctic highland (−18°C, permafrost)",
    3:  "Cold boreal (−10°C, permafrost 80%)",
    1:  "Cold karst highland (−8°C, karst 76%)",
    13: "Northern peatlands (−7°C, wetland 44%)",
    11: "Mountain permafrost (−5°C, steep)",
    12: "Subarctic wetlands (+1°C, wetland 52%)",
    10: "Cool temperate lowlands (+5°C)",
    16: "Arid mountain interior (+11°C, arid)",
    17: "Large river systems (+15°C, Q=5693 m³/s)",
    18: "Warm humid karst (+15°C, karst 66%)",
    8:  "Regulated rivers (+15°C, reservoir)",
    15: "Tropical wet mountains (+17°C, steep)",
    14: "Tropical flat lowlands (+19°C, slope 1°)",
    9:  "Warm arid floodplains (+22°C, wetland 37%)",
    7:  "Hot arid karst (+22°C, karst 81%)",
    6:  "Warm semi-arid (+23°C)",
    4:  "Tropical humid (+23°C, precip 1154mm)",
    2:  "Hyperarid desert (+23°C, precip 37mm)",
    5:  "Hot humid floodplains (+25°C, wetland 54%)",
    0:  "Hot wet tropics (+26°C, precip 2049mm)",
}

# Global cluster sizes (baseline)
global_counts = assignments.km_cluster.value_counts().sort_index()
global_pct = (global_counts / global_counts.sum() * 100).round(2)
print(f"Loaded {len(assignments):,} basin assignments, {len(CLUSTER_LABELS)} clusters")

In [ ]:
# Cell 3 — WH Cities: assign to clusters via existing basin_id

with conn.cursor() as cur:
    cur.execute("""
        SELECT id, city,
               ST_X(geom::geometry) AS lon,
               ST_Y(geom::geometry) AS lat,
               basin_id
        FROM gaz.wh_cities
        WHERE geom IS NOT NULL
    """)
    rows = cur.fetchall()
    whc = pd.DataFrame(rows, columns=['id','city','lon','lat','basin_id'])

# Map basin_id → km_cluster
whc['km_cluster'] = whc['basin_id'].map(
    lambda x: basin_to_cluster.get(int(x)) if pd.notna(x) else None
)
print(f"WH Cities: {len(whc)} total, {whc.km_cluster.notna().sum()} with cluster assigned")
print(whc[whc.km_cluster.isna()][['city','lat','lon']].head(10))

In [ ]:
# Cell 4 — D-PLACE: bulk nearest-basin spatial lookup
#
# Uses KNN geography operator (<->) against basin08.geog index.
# For each society, finds the nearest basin centroid (by geography distance).
# This is approximate — ignores up_area weighting used in the sandbox —
# but is appropriate for the coverage characterization purpose of Task 6.
# ~6400 lookups with a geography index: expect 30–90 seconds.

print("Fetching D-PLACE societies...")
with conn.cursor() as cur:
    cur.execute("""
        SELECT id, name, latitude::float, longitude::float, region
        FROM dplace.societies
        WHERE latitude IS NOT NULL AND longitude IS NOT NULL
    """)
    rows = cur.fetchall()
    dplace = pd.DataFrame(rows, columns=['id','name','lat','lon','region'])

print(f"D-PLACE societies with coordinates: {len(dplace):,}")

print("Running bulk nearest-basin lookup (may take ~60 seconds)...")
with conn.cursor() as cur:
    cur.execute("""
        SELECT s.id, b.hybas_id
        FROM dplace.societies s
        CROSS JOIN LATERAL (
            SELECT hybas_id
            FROM public.basin08
            ORDER BY geom::geography <->
                ST_SetSRID(ST_MakePoint(s.longitude::float, s.latitude::float), 4326)::geography
            LIMIT 1
        ) b
        WHERE s.latitude IS NOT NULL AND s.longitude IS NOT NULL
    """)
    soc_basins = dict(cur.fetchall())

print(f"Basin assignments retrieved: {len(soc_basins):,}")

dplace['hybas_id'] = dplace['id'].map(soc_basins)
dplace['km_cluster'] = dplace['hybas_id'].map(
    lambda x: basin_to_cluster.get(int(x)) if pd.notna(x) else None
)
print(f"D-PLACE with cluster assigned: {dplace.km_cluster.notna().sum():,}")

In [ ]:
# Cell 5 — Distribution comparison table
#
# For each cluster: global %, D-PLACE %, WH Cities %.
# Sorted by global prevalence (most common environmental types first).

dplace_counts = dplace['km_cluster'].dropna().astype(int).value_counts()
whc_counts    = whc['km_cluster'].dropna().astype(int).value_counts()

rows = []
for c in range(20):
    g = global_counts.get(c, 0)
    d = dplace_counts.get(c, 0)
    w = whc_counts.get(c, 0)
    rows.append({
        'cluster': c,
        'label': CLUSTER_LABELS.get(c, f'Cluster {c}'),
        'global_n':   g,
        'global_pct': round(g / global_counts.sum() * 100, 2),
        'dplace_n':   d,
        'dplace_pct': round(d / dplace_counts.sum() * 100, 2),
        'whc_n':      w,
        'whc_pct':    round(w / whc_counts.sum() * 100, 2) if whc_counts.sum() > 0 else 0,
    })

dist = pd.DataFrame(rows).sort_values('global_pct', ascending=False)
print(dist[['label','global_pct','dplace_pct','whc_pct']].to_string(index=False))

In [ ]:
# Cell 6 — Over/under-representation ratios
#
# ratio = dataset_pct / global_pct
# >2.0: strongly over-represented in scholarship
# <0.5: strongly under-represented
# ~1.0: proportional to global prevalence

dist['dplace_ratio'] = (dist['dplace_pct'] / dist['global_pct'].replace(0, np.nan)).round(2)
dist['whc_ratio']    = (dist['whc_pct']    / dist['global_pct'].replace(0, np.nan)).round(2)

print("=== D-PLACE over-represented (ratio > 2.0) ===")
over_d = dist[dist.dplace_ratio > 2.0].sort_values('dplace_ratio', ascending=False)
print(over_d[['label','global_pct','dplace_pct','dplace_ratio']].to_string(index=False))

print("\n=== D-PLACE under-represented (ratio < 0.5) ===")
under_d = dist[dist.dplace_ratio < 0.5].sort_values('dplace_ratio')
print(under_d[['label','global_pct','dplace_pct','dplace_ratio']].to_string(index=False))

print("\n=== WH Cities over-represented (ratio > 2.0) ===")
over_w = dist[dist.whc_ratio > 2.0].sort_values('whc_ratio', ascending=False)
print(over_w[['label','global_pct','whc_pct','whc_ratio']].to_string(index=False))

print("\n=== WH Cities under-represented (ratio < 0.5) ===")
under_w = dist[dist.whc_ratio < 0.5].sort_values('whc_ratio')
print(under_w[['label','global_pct','whc_pct','whc_ratio']].to_string(index=False))

In [ ]:
# Cell 7 — Bar chart: representation ratios by cluster
#
# log2 ratio (0 = proportional, +1 = 2× over, −1 = 2× under)
# Sorted by global prevalence (most common types at top).

plot_df = dist.sort_values('global_pct', ascending=True).copy()
plot_df['dplace_log2'] = np.log2(plot_df['dplace_ratio'].replace(0, np.nan))
plot_df['whc_log2']    = np.log2(plot_df['whc_ratio'].replace(0, np.nan))

labels = [f"{row.label[:45]}  ({row.global_pct:.1f}%)" for _, row in plot_df.iterrows()]

fig, axes = plt.subplots(1, 2, figsize=(16, 9), sharey=True)

for ax, col, title, color in [
    (axes[0], 'dplace_log2', 'D-PLACE societies', 'steelblue'),
    (axes[1], 'whc_log2',    'WH Cities',          'darkorange'),
]:
    vals = plot_df[col].fillna(-4)  # missing = extreme under
    colors = ['tomato' if v > 0 else 'steelblue' if color == 'steelblue' else 'darkorange'
              for v in vals]
    ax.barh(range(len(plot_df)), vals, color=colors, alpha=0.8)
    ax.axvline(0, color='black', lw=0.8, ls='--')
    ax.axvline(1,  color='gray', lw=0.5, ls=':')
    ax.axvline(-1, color='gray', lw=0.5, ls=':')
    ax.set_xlabel('log₂(dataset % / global %)', fontsize=9)
    ax.set_title(f'{title}\nlog₂ representation ratio', fontsize=10)
    ax.tick_params(labelsize=7)
    ax.set_xlim(-4.5, 4.5)
    ax.text(1.05, -1, '2×\nover', transform=ax.get_xaxis_transform(),
            fontsize=7, color='gray', ha='center')
    ax.text(-1.05, -1, '2×\nunder', transform=ax.get_xaxis_transform(),
            fontsize=7, color='gray', ha='center')

axes[0].set_yticks(range(len(plot_df)))
axes[0].set_yticklabels(labels, fontsize=6.5)

plt.suptitle('Scholarship coverage vs. global basin distribution (log₂ ratio)',
             fontsize=11, y=1.01)
plt.tight_layout()
plt.savefig(f'{out_dir}/06_representation_ratios.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved 06_representation_ratios.png")

In [ ]:
# Cell 8 — Global map: D-PLACE and WH Cities overlaid on k-means base
#
# Background: k-means cluster centroids (small, faint)
# Foreground: D-PLACE societies (blue dots) and WH Cities (orange stars)

cmap = plt.get_cmap('tab20', 20)

fig, ax = plt.subplots(figsize=(18, 9))
ax.set_facecolor('#1a1a2e')
fig.patch.set_facecolor('#1a1a2e')

# Background: basin centroids by cluster (faint)
for c in range(20):
    mask = assignments['km_cluster'] == c
    lons = assignments.loc[mask, 'hybas_id'].map(
        lambda hid: None)  # centroids not in assignments — use basin lon/lat
    
# Load centroid coordinates from the cluster assignments (we need lon/lat)
# Fetch from DB instead
with conn.cursor() as cur:
    cur.execute("""
        SELECT hybas_id,
               ST_X(ST_Centroid(geom)) AS lon,
               ST_Y(ST_Centroid(geom)) AS lat
        FROM public.basin08
    """)
    coord_rows = cur.fetchall()
coord_df = pd.DataFrame(coord_rows, columns=['hybas_id','lon','lat'])
coord_df['km_cluster'] = coord_df['hybas_id'].map(basin_to_cluster)

# Plot background basins
for c in range(20):
    mask = coord_df['km_cluster'] == c
    ax.scatter(coord_df.loc[mask,'lon'], coord_df.loc[mask,'lat'],
               s=0.15, color=cmap(c), alpha=0.3, linewidths=0)

# D-PLACE societies
ax.scatter(dplace['lon'], dplace['lat'],
           s=4, color='white', alpha=0.7, linewidths=0,
           label=f'D-PLACE ({len(dplace):,})', zorder=5)

# WH Cities
ax.scatter(whc['lon'], whc['lat'],
           s=20, color='gold', marker='*', alpha=0.95, linewidths=0.3,
           edgecolors='white', label=f'WH Cities ({len(whc)})', zorder=6)

ax.set_xlim(-180, 180)
ax.set_ylim(-60, 85)
ax.set_title('D-PLACE societies and WH Cities on k-means environmental clusters (L8)',
             color='white', fontsize=11)
ax.tick_params(colors='white')
for spine in ax.spines.values():
    spine.set_edgecolor('gray')

legend = ax.legend(loc='lower left', fontsize=8, framealpha=0.5,
                   labelcolor='white', facecolor='#1a1a2e', edgecolor='gray')

plt.tight_layout()
plt.savefig(f'{out_dir}/06_coverage_map.png', dpi=150, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()
print("Saved 06_coverage_map.png")

In [ ]:
# Cell 9 — Save distribution and ratio tables

dist.to_csv(f'{out_dir}/06_coverage_distribution.csv', index=False)
print("Saved 06_coverage_distribution.csv")

# Also save D-PLACE with cluster assignments for future use
dplace[['id','name','lat','lon','region','hybas_id','km_cluster']].to_csv(
    f'{out_dir}/06_dplace_cluster_assignments.csv', index=False)
print("Saved 06_dplace_cluster_assignments.csv")

print(f"\nSummary:")
print(f"  Global basins:        {global_counts.sum():,}")
print(f"  D-PLACE assigned:     {dplace.km_cluster.notna().sum():,} / {len(dplace):,}")
print(f"  WH Cities assigned:   {whc.km_cluster.notna().sum()} / {len(whc)}")